In [1]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)
tf.random.set_seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

In [2]:
# Try to make GPU memory allocation dynamic, so it doesn't slow down/crash
physical_devices = tf.config.list_physical_devices('GPU')
try:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    # config = tf.compat.v1.ConfigProto()
    # config.gpu_options.allow_growth = True
    # session = tf.compat.v1.Session(config=config)
except:
    print('oh no.')
  # Invalid device or cannot modify virtual devices once initialized.
    pass

In [3]:
# Check and test that TensorFlow can use GPU/CUDA cores

# tf.debugging.set_log_device_placement(True)

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Create some tensors
a = tf.constant([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
b = tf.constant([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
c = tf.matmul(a, b)

tf.debugging.set_log_device_placement(False)
print(c)

Num GPUs Available:  1
tf.Tensor(
[[22. 28.]
 [49. 64.]], shape=(2, 2), dtype=float32)


In [4]:
physical_devices

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [5]:
#The goal of this project is to use a Generative Adversarial Network (GAN) to generate supplemental data for histopathological brain tumor samples, providing a solution to the relative scarcity of publicly available brain tumor data for machine learning and other
# scientific endeavors.
#
# For the time being, we focus on creating a proof of concept that works on smaller iomages while we figure out how to work around the large filesizes of the Digital Brain Tunmour Atlas' histopathological slide images.
# The GAN should be context-agnostic, so it should work on any images of brain tumors.
# As such, for the proof of concept model, we use images from Kaggle's Brain Tumor MRI Dataset, which combines several datasets into one place: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset?resource=download
# This data must be preprocessed, since images from different sources are of difference sizes.

In [6]:
# TO-DO LIST
#
# MAIN PROJECT: Import .csv of the data, identify which 3/4 types of brain tumors are most common.
#                     Related --> find out how to import .csv as pandas dataframe for easier working.
#               Determine how/where to save data????? The images are huge, 3TiB in total. Even just the ones we care about will take up a huge amount of space.
#               Determine and implement the best quantitative evaluation metric for this project.
#
# PROOF OF CONCEPT: Import the jpg images, crop

In [7]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install imutils

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import numpy as np 
from tqdm import tqdm
import cv2
# import os
import imutils

# change this to wherever your images are located
DATASET_PATH = r'C:/Users/Aihara/Desktop/Temporary Jupter Stuff/Kaggle - Brain Tumor MRI Dataset'


def crop_img(img):
	"""
	Finds the extreme points on the image and crops the rectangular out of them
	"""
	gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
	gray = cv2.GaussianBlur(gray, (3, 3), 0)

	# threshold the image, then perform a series of erosions +
	# dilations to remove any small regions of noise
	thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
	thresh = cv2.erode(thresh, None, iterations=2)
	thresh = cv2.dilate(thresh, None, iterations=2)

	# find contours in thresholded image, then grab the largest one
	cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
	cnts = imutils.grab_contours(cnts)
	c = max(cnts, key=cv2.contourArea)

	# find the extreme points
	extLeft = tuple(c[c[:, :, 0].argmin()][0])
	extRight = tuple(c[c[:, :, 0].argmax()][0])
	extTop = tuple(c[c[:, :, 1].argmin()][0])
	extBot = tuple(c[c[:, :, 1].argmax()][0])
	ADD_PIXELS = 0
	new_img = img[extTop[1]-ADD_PIXELS:extBot[1]+ADD_PIXELS, extLeft[0]-ADD_PIXELS:extRight[0]+ADD_PIXELS].copy()
	
	return new_img
	
if __name__ == "__main__":
	training = DATASET_PATH + "/Training"
	testing = DATASET_PATH + "/Testing"
	training_dir = os.listdir(training)
	testing_dir = os.listdir(testing)
	IMG_SIZE = 256

	for dir in training_dir:
		save_path = 'cleaned/Training/'+ dir
		path = os.path.join(training,dir)
		image_dir = os.listdir(path)
		for img in image_dir:
			image = cv2.imread(os.path.join(path,img))
			new_img = crop_img(image)
			new_img = cv2.resize(new_img,(IMG_SIZE,IMG_SIZE))
			if not os.path.exists(save_path):
				os.makedirs(save_path)
			cv2.imwrite(save_path+'/'+img, new_img)
	
	for dir in testing_dir:
		save_path = 'cleaned/Testing/'+ dir
		path = os.path.join(testing,dir)
		image_dir = os.listdir(path)
		for img in image_dir:
			image = cv2.imread(os.path.join(path,img))
			new_img = crop_img(image)
			new_img = cv2.resize(new_img,(IMG_SIZE,IMG_SIZE))
			if not os.path.exists(save_path):
				os.makedirs(save_path)
			cv2.imwrite(save_path+'/'+img, new_img)

In [ ]:
# # Just double checking that the input images really are 0-255

# temp_glioma_files = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\Kaggle - Brain Tumor MRI Dataset\\Training\\glioma"
# current_img = "Te-gl_0010.jpg"
# # print(os.path.join(temp_glioma_files, current_img))
# # print(os.listdir(temp_glioma_files))

# img_path = os.path.join(temp_glioma_files, os.listdir(temp_glioma_files)[0])
# # asdf = cv2.imread(os.path.join(temp_glioma_files, current_img), 0)

# if os.path.isfile(img_path):
#     print(img_path)
#     img1 = cv2.imread(img_path, 0)
#     img1 = cv2.applyColorMap(img1, cv2.COLORMAP_BONE)
# else:
#     print ("The file " + img_path + " does not exist.")

# print(img1[150,177])
# plt.imshow(img1)

In [ ]:
# Code to load images borrowed from https://github.com/masoudnick/Brain-Tumor-MRI-Classification/blob/main/Brain_Tumor_Classification.ipynb

# Change these to where your cleaned data are located
CLEANED_TRAINING_PATH = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\cleaned\\Training"
CLEANED_TESTING_PATH = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\cleaned\\Testing"

labels = ['glioma', 'meningioma', 'notumor', 'pituitary']

x_train = [] # training images.
y_train  = [] # training labels.
x_test = [] # testing images.
y_test = [] # testing labels.

image_size = 200


for label in labels:
    trainPath = os.path.join(CLEANED_TRAINING_PATH,label)
    for file in tqdm(os.listdir(trainPath)):
        image = cv2.imread(os.path.join(trainPath, file),0) # load images in gray.
        image = cv2.bilateralFilter(image, 2, 50, 50) # remove images noise.
        image = cv2.applyColorMap(image, cv2.COLORMAP_BONE) # produce a pseudocolored image.
        image = cv2.resize(image, (image_size, image_size)) # resize images into 200*200.
        x_train.append(image)
        y_train.append(labels.index(label))
    
    testPath = os.path.join(CLEANED_TESTING_PATH,label)
    for file in tqdm(os.listdir(testPath)):
        image = cv2.imread(os.path.join(testPath, file),0)
        image = cv2.bilateralFilter(image, 2, 50, 50)
        image = cv2.applyColorMap(image, cv2.COLORMAP_BONE)
        image = cv2.resize(image, (image_size, image_size))
        x_test.append(image)
        y_test.append(labels.index(label))


x_train = np.array(x_train, dtype=np.float32) / 255.0 # normalize Images into range 0 to 1.
x_test = np.array(x_test, dtype=np.float32) / 255.0

# By default, Tensorflow uses float32.You have to convert your data to tf.float32.
# X = tf.cast(yourDATA, tf.float32) 

# x_train = tf.cast(x_train, tf.float32)
# x_test = tf.cast(x_test, tf.float32)

print(x_train.shape)
print(x_test.shape)




In [ ]:
print(type(x_train[0,0,0,0]))

In [ ]:
# These images are (200, 200, 3) each, since they have 3 channels (assuming they're Red, Blue, and Green channels).

In [ ]:
# Checking on the images

images = [x_train[i] for i in range(15)]
fig, axes = plt.subplots(3, 5, figsize = (10, 10))
axes = axes.flatten()
for img, ax in zip(images, axes):
    ax.imshow(img)
plt.tight_layout()
plt.show()

In [ ]:
print(x_train[0][:,:,0].shape)
fig, axes = plt.subplots(1, 1, figsize = (10, 10))
axes.imshow(x_train[0])
print(np.max(x_train[0]))
plt.show()

In [ ]:
# DCGAN code taken from HOA #8 - will tweak as necessary to serve as a basis for this work.
# The DCGAN seemed to have better initial and overall performance as compared to the standard GAN, so we use it as the baseline here.

In [ ]:
def plot_multiple_images(images, n_cols=None):
    n_cols = n_cols or len(images)
    n_rows = (len(images) - 1) // n_cols + 1
    if images.shape[-1] == 1:
        images = np.squeeze(images, axis=-1)
    plt.figure(figsize=(n_cols, n_rows))
    for index, image in enumerate(images):
        plt.subplot(n_rows, n_cols, index + 1)
        plt.imshow(image)#, cmap='bone')
        # plt.imshow(image[:,:,::-1])
        plt.axis("off")

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)


# codings_size = 16
codings_size = 32

generator = keras.models.Sequential([
    keras.layers.Dense(50 * 50 * 128, input_shape=[codings_size]),
    keras.layers.Reshape([50, 50, 128]),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding="SAME",
                                 activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2DTranspose(3, kernel_size=5, strides=2, padding="SAME",
                                 activation="tanh"),
])

discriminator = keras.models.Sequential([
    keras.layers.Conv2D(64, kernel_size=5, strides=2, padding="SAME",
                        activation=keras.layers.LeakyReLU(0.2),
                        input_shape=[200, 200, 3]),
    keras.layers.BatchNormalization(),
    # keras.layers.Dropout(0.4),
    keras.layers.Conv2D(128, kernel_size=5, strides=2, padding="SAME",
                        activation=keras.layers.LeakyReLU(0.2)),
    keras.layers.BatchNormalization(),
    # keras.layers.Dropout(0.4),
    keras.layers.Flatten(),
    keras.layers.Dense(1, activation="sigmoid")
])

gan = keras.models.Sequential([generator, discriminator])

In [ ]:
generator.summary()
discriminator.summary()
gan.summary()

In [ ]:
# Since the discriminator is a binary classifier (real/fake image), we can set the loss to binary_crossentropy.
# Similarly, since the GAN as a whole is a binary classifier, we can do the same.
# 
# Once the GAN is created, we must appropriately compile/determine which parts of the model are trainable or not.
# Since "trainable" is only taken into account while compiling, we first compile the discriminator assuming it will be trained. Then, we switch it so that it cannot be trained once the complete GAN is compiled.  
# This way, when we train the GAN, we can separate the training of the discriminator and the generator during each pass - the discriminator is first trained on 

In [ ]:
discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")
discriminator.trainable = False
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

In [ ]:
# In order to train a GAN, we must use a custom training loop similar to the one defined in the HOA/textbook.
# To do this, the training is split into two phases for each training pass/epoch.
# In phase 1, the discriminator is trained on a a combined dataset of real images and images built by the generator using Gaussian noise as input.
# In phase 2, the generator is trained by feeding Gaussian noise to the GAN itself. Now, the images from the generator are tested using the newly-trained discriminator from phase 1. 

In [ ]:
batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(1000)
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)

In [ ]:
# Reshape and rescale the data to account for the tanh activation function in the generator.
# I think we need to undo this rescaling when plotting/displaying the generated images, since some will have negative values that get clipped when plotted by imshow().
x_train_dcgan = x_train.reshape(-1, 200, 200, 3) * 2. - 1. 

In [ ]:
def train_gan(gan, dataset, batch_size, codings_size, n_epochs=50):
    generator, discriminator = gan.layers
    exp_replay = []
    
    for epoch in range(n_epochs):
        print("Epoch {}/{}".format(epoch + 1, n_epochs))
        for X_batch in dataset:
            
            # PHASE 1 - TRAINING THE DISCRIMINATOR
            
            # Generating fake images for training the discriminator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            generated_images = generator(noise)
            
            # Combined dataset of both real and newly generated fake images 
            X_fake_and_real = tf.concat([generated_images, X_batch], axis=0)
            y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)
            
            # Actually training the discriminator on the above data.
            discriminator.trainable = True
            discriminator.train_on_batch(X_fake_and_real, y1)
            
            # PHASE 2 - TRAINING THE GENERATOR
            
            # Generating noise to feed into the GAN
            noise = tf.random.normal(shape=[batch_size, codings_size])
            y2 = tf.constant([[1.]] * batch_size)
            
            # Actually training the generator by having the newly-trained discriminator classify its attempts as real/fake images. 
            discriminator.trainable = False
            gan.train_on_batch(noise, y2)         
            
            ######################################################################################
            # Experience Replay. Not sure if this is working properly.
            r_index = np.random.randint(batch_size)
            # exp_replay.append([generated_images[r_index], labels[r_index], gene_labels[r_index]])
            exp_replay = np.append(exp_replay, generated_images[r_index])
            
            if len(exp_replay) == batch_size:
                # gen_img = np.array([p[0] for p in exp_replay])
                # labels = np.array([p[1] for p in exp_replay])
                # gen_labels = np.array([p[2] for p in exp_replay])
                replay_X_fake_and_real = tf.concat([exp_replay, X_batch], axis=0)
                y_replay = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)
                
                # Train discriminator or combination of real images and newer/older generated images.
                discriminator.trainable = True
                exprep_loss_gen = discriminator.train_on_batch(replay_X_fake_and_real, y_replay)
                discriminator.trainable = False
                
                # Remove older generated images from the buffer list.
                exp_replay = np.delete(exp_replay, [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15])
                
            ######################################################################################
        
        plot_multiple_images(generated_images, 8)
        plt.show()

In [ ]:
# testarr = np.array([1,2,3,4,5,6,7,8,9,10])
# np.delete(testarr,[0,1,2,3,4])

In [ ]:
train_gan(gan, dataset, batch_size, codings_size, n_epochs=10)

In [ ]:
# Notes:
#
# ARe the images diverse enough? Maybe try to apply some more corrections to the model
#     -> Experience Replay (check that this is implemented corectly)
# 1) Minibatch std. dev. layer
# 2) Equalized learning rate
# 3) Pixelwise normalization layer

In [ ]:
# noise = tf.random.normal(shape=[batch_size, codings_size])
# generated_images = generator(noise)
# generated_images.shape
# # generated_images = tf.math.l2_normalize(generated_images, axis=())

In [ ]:
# plt.imshow(generated_images[0].numpy()[:,:,::-1], cmap = 'gray')
# plt.show()
# # plt.imshow(generated_images[0].numpy(), cmap = 'bone')